# fase 4: analise exploratoria

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/processed/hourly_features.csv")
df.shape

## 0. vies de amostragem

taxa de is_empty por readings_count, estratificada por hour_of_day (pivo hour_of_day x readings_count). a checagem simples (sem estratificar por hora) e confundida: readings_count varia com a hora do dia porque a coleta e do sistema inteiro de uma vez, entao correlacionar direto mistura efeito de hora com efeito de amostragem.

In [ ]:
pivot_is_empty = df.pivot_table(index="hour_of_day", columns="readings_count", values="is_empty", aggfunc="mean")
print(pivot_is_empty.round(3))

within_hour_corr = df.groupby("hour_of_day").apply(lambda g: g["readings_count"].corr(g["is_empty"]))
print("\ncorrelacao readings_count x is_empty, dentro de cada hora:")
print(within_hour_corr.round(3))
print("\nmedia das correlacoes por hora:", round(within_hour_corr.mean(), 3))
print("horas com correlacao negativa:", int((within_hour_corr < 0).sum()), "de", len(within_hour_corr))

dentro da mesma hora, a taxa NAO sobe com readings_count -- cai, de forma consistente: 23 das 24 horas tem correlacao negativa (media -0.161, entre -0.08 e -0.22). so a hora 3 e levemente positiva (+0.04), com poucas observacoes nas colunas mais altas.

controlar por hora tornou o efeito mais forte, nao mais fraco: a correlacao global nao-estratificada era -0.115, dentro da hora fica -0.161 em media.

conclusao: nao ha vies de amostragem. a relacao e inversa e consistente nas 24 horas -- o oposto do que geraria contaminacao. is_empty continua sendo o alvo, sem trocar para empty_share.

hipotese (nao testada) pra inversao: o feed provavelmente atualiza quando o estado da estacao muda (bike sai ou chega), entao estacao vazia parada gera menos movimento e menos leituras por hora. nao investigado nesta fase.

## 1. mapa de calor: estacao x hora do dia

so as 30 estacoes com maior taxa media de is_empty, ordenadas da pior pra melhor.

In [ ]:
top_stations = df.groupby("name")["is_empty"].mean().sort_values(ascending=False).head(30).index

pivot = df[df["name"].isin(top_stations)].pivot_table(
    index="name", columns="hour_of_day", values="is_empty", aggfunc="mean"
)
pivot = pivot.loc[top_stations]  # mantem a ordem da pior pra melhor estacao

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(pivot, cmap="Reds", ax=ax, cbar_kws={"label": "taxa de is_empty"})
ax.set_title("Taxa de is_empty por estacao e hora do dia (30 estacoes com maior taxa)")
ax.set_xlabel("hora do dia (hora local de Dublin)")
ax.set_ylabel("estacao")
plt.tight_layout()
plt.savefig("../reports/heatmap_station_hour.png", dpi=150)
plt.show()

## 2. taxa por hora do dia: dia util vs fim de semana

In [ ]:
by_hour_daytype = df.groupby(["hour_of_day", "is_weekend"])["is_empty"].mean().unstack()
by_hour_daytype.columns = ["dia util", "fim de semana"]

fig, ax = plt.subplots(figsize=(10, 6))
by_hour_daytype.plot(ax=ax, marker="o")
ax.set_title("Taxa de is_empty por hora do dia: dia util vs fim de semana")
ax.set_xlabel("hora do dia (hora local de Dublin)")
ax.set_ylabel("taxa de is_empty")
ax.set_xticks(range(0, 24, 2))
ax.legend(title="")
plt.tight_layout()
plt.savefig("../reports/hourly_weekday_weekend.png", dpi=150)
plt.show()

## 3. efeito da chuva

In [ ]:
bins = [-0.01, 0, 0.5, 2, df["prcp"].max()]
labels = ["0mm", "0-0.5mm", "0.5-2mm", ">2mm"]
df["prcp_bin"] = pd.cut(df["prcp"], bins=bins, labels=labels)

rain_stats = df.groupby("prcp_bin", observed=True).agg(
    is_empty_rate=("is_empty", "mean"),
    n_hours=("hour_utc", "nunique"),  # horas distintas, nao linhas estacao-hora
)
print(rain_stats)

fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.bar(rain_stats.index.astype(str), rain_stats["is_empty_rate"], color="steelblue")
for bar, n in zip(bars, rain_stats["n_hours"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005, f"n={n} horas", ha="center")
ax.set_title("Taxa de is_empty por faixa de precipitacao")
ax.set_xlabel("faixa de precipitacao (prcp)")
ax.set_ylabel("taxa de is_empty")
plt.tight_layout()
plt.savefig("../reports/rain_effect.png", dpi=150)
plt.show()

## 4. distribuicao da taxa de is_empty por estacao (115 estacoes)

In [ ]:
station_rate = df.groupby("name")["is_empty"].mean()

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(station_rate, bins=15, color="darkorange", edgecolor="black")
ax.set_title("Distribuicao da taxa de is_empty entre as 115 estacoes")
ax.set_xlabel("taxa media de is_empty")
ax.set_ylabel("numero de estacoes")
plt.tight_layout()
plt.savefig("../reports/station_distribution.png", dpi=150)
plt.show()

station_rate.describe()

## achados

1. **padrao de commute nas estacoes.** dia util tem dois picos de taxa de is_empty, ~08h (34%) e ~17-18h (32%); fim de semana tem um plato largo e mais baixo entre 11h-16h (~28-29%), sem os dois picos. bate com bike saindo de estacao residencial de manha e voltando a noite, contra uso de lazer no fim de semana.

2. **poucas estacoes concentram quase toda a emptiness.** a media geral e 26.5%, mas a pior estacao (HARDWICKE PLACE) fica vazia em 75% das horas, e as 3 piores (HARDWICKE PLACE, PARNELL SQUARE NORTH, ECCLES STREET EAST) passam de 65%. sao as candidatas obvias a rebalanceamento prioritario.

3. **is_empty nao esta contaminado pela cadencia.** correlacao entre readings_count e is_empty e -0.11 (cai, nao sobe, com mais leituras) -- o oposto do vies temido no item 0. nao ha necessidade de trocar por empty_share nesta fase.

## chuva

efeito praticamente nulo, e na direcao contraria a esperada: taxa cai de 26.5% (sem chuva) pra 24.2% (>2mm), uma diferenca de ~2 pontos percentuais. nao da pra confiar nesse numero: a faixa >2mm tem so 6 horas distintas em 6 meses de dado.

## distribuicao por estacao

continua e assimetrica a direita, nao bimodal. a maioria das estacoes fica entre 4% e 25% de taxa de is_empty, com uma cauda longa de poucas estacoes chegando a 40-75%. nao ha dois agrupamentos separados -- e uma decadencia gradual das estacoes tranquilas ate as poucas estacoes-problema.